# S2 · NB2 — the optimism bias

**The centrepiece.** An oracle ceiling computed from the same seed it routes is
optimistically biased: it partly routes on that model's own noise, which no
deployable router could have.

```
in-seed   oracle : score from seed i  -> routes seed i's model   (optimistic)
cross-seed oracle: score from seed j  -> routes seed i's model   (honest)
optimism bias    = in-seed - cross-seed            at matched FLOPs
```

Answers **R3**, **R4** and **R5**. CPU only — per-exit predictions are already
in the parquet, so no model is loaded.

In [ ]:
# ============================================================================
# CELL 1 -- unpack the library.  Runs in every notebook.  No network.
# ============================================================================
# Writes two files into the working directory and imports them:
#
#   msc_lib.py    ce91e6fcd51f   the pipeline: data, zoo, training, measurement
#   msc_core.py   2cc4ba5e0935   the reference maths: the MSC definition and
#                                    every statistic in the paper
#
# Both are GENERATED from src/ by build_notebooks_in100.py. Editing the base64
# below does nothing that survives a rebuild -- edit src/msc_lib.py instead.
#
# NOTHING IS INSTALLED HERE. This pipeline runs offline; the packages must
# already be present (see requirements.txt). A missing one is reported by name
# with what it costs you, rather than silently pip-installing on a machine that
# may have no network.
import base64, os, sys
from pathlib import Path

# Offline guards must be set BEFORE anything that might fetch is imported.
os.environ.setdefault('MSC_OFFLINE', '1')

WORK = Path.cwd()
_LIB = (
    '...',
    'NTFmIgo=',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]          # force reimport if this cell is re-run
import importlib
importlib.invalidate_caches()

_MISSING = []
for _pkg, _why in (('torch', 'everything'),
                   ('torchvision', 'resnet/vgg/shufflenet/swin'),
                   ('numpy', 'everything'), ('pandas', 'every table'),
                   ('pyarrow', 'per_sample/*.parquet -- the science'),
                   ('yaml', 'config.yaml per run'),
                   ('scipy', 'Spearman = Q1 and Q3'),
                   ('sklearn', 'Q4 delta-R2, Q2 PCA'),
                   ('psutil', 'host telemetry columns'),
                   ('pynvml', 'GPU power -- energy columns are NA without it'),
                   ('fvcore', 'FLOPs. rho is DEFINED in FLOPs.')):
    try:
        __import__(_pkg)
    except ImportError:
        _MISSING.append(f'{_pkg:12s} {_why}')
if _MISSING:
    print('MISSING PACKAGES -- install these, then restart the kernel:')
    for _m in _MISSING:
        print('   ', _m)
    raise SystemExit('see requirements.txt')

import msc_lib as M
import torch

# D-62. Prove the module that LOADED is the module that SHIPPED.
#
# Twice now a fix was applied, verified, regenerated -- and the run failed with
# the identical error, because the code executing was not the code on disk.
# Jupyter keeps an imported module until something removes it, and any object
# built from the old module (a Session, say) keeps its old functions even after
# a reimport. There was no mechanism that could tell the difference, so the
# evidence looked like "the fix does not work" when it was "the fix never ran".
#
# Rule 5: a cache must answer "is what I have still VALID", not "do I have
# something". The stamp is written into the bytes this cell decodes, so it
# cannot drift from them.
_want = 'ce91e6fcd51f'
_got = getattr(M, '__MSC_BUILD__', None)
if _got != _want:
    raise RuntimeError(
        f"STALE msc_lib: this notebook ships build {_want} but the imported "
        f"module reports {_got}.\n"
        f"  loaded from: {getattr(M, '__file__', '?')}\n"
        f"  Restart the kernel (Kernel -> Restart) and run all cells. Objects "
        f"created before a reimport keep the OLD code even after this cell "
        f"rewrites the file (D-62).")
# D-68. Is this NOTEBOOK current with the repository?
#
# The check above proves the module matches the notebook. It CANNOT catch a
# stale notebook, because both sides come from the same .ipynb -- they always
# agree with each other and can be arbitrarily old together.
#
# Jupyter saves an open notebook on run. So regenerating NB3 on disk while it
# sits open in a tab means the tab's copy wins the moment you run it: the fixed
# notebook is silently replaced by the one that was open, and the fix appears
# not to have been applied. That happened here -- NB3 was regenerated with
# `done_fn=sess.measured, stage='measure'`, and the version that ran had
# neither.
#
# The repository source is the authority. If it has moved on, this notebook is
# stale and must be reopened, not re-run.
_repo = WORK.parent / 'src' / 'msc_lib.py'
if _repo.exists():
    import hashlib as _h
    _repo_sha = _h.sha256(_repo.read_bytes()).hexdigest()[:12]
    if _repo_sha != _want:
        raise RuntimeError(
            f"STALE NOTEBOOK: this file embeds msc_lib {_want}, but "
            f"src/msc_lib.py is {_repo_sha}.\n"
            f"  You are running an older copy of this notebook. Jupyter saves "
            f"an open notebook when you run it, so an open tab silently "
            f"overwrites a regenerated file.\n"
            f"  FIX: close this notebook WITHOUT saving, run "
            f"`python build_notebooks_in100.py`, then reopen it (D-68).")
    print(f'msc_lib build {_got} verified, and current with src/')
else:
    print(f'msc_lib build {_got} verified (repo source not visible)')

print(f'msc_lib {M.__version__}   torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name}  {_p.total_memory/2**30:.1f} GiB  sm_{_p.major}{_p.minor}')
else:
    print('  *** NO CUDA. A CPU-only torch trains at roughly 1/200th speed')
    print('  *** while reporting entirely plausible numbers. Fix this first.')

In [ ]:
# ============================================================================
# CELL 2 -- WHERE EVERYTHING LIVES
# ============================================================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine gets `msc_data/in100` and `msc_results`.
#
# The previous version defaulted to r'D:\msc_data\in100'. There is no D:
# drive here, and the failure was
#
#     FileNotFoundError: [WinError 3] The system cannot find the path
#     specified: 'D:\'
#
# forty lines deep inside pathlib, naming neither the setting nor the file that
# had to change. A default that names a drive letter is wrong on any machine
# without that letter (D-44).
#
# Set them explicitly if you want somewhere specific. Both are checked below by
# WRITING A PROBE FILE AND READING IT BACK -- os.access lies on Windows shares.
#
#   data     ~26 GB   the packed dataset, read-only after NB1
#   results ~120 GB   every run. Nothing here is ever deleted.

DATA_DIR = None      # e.g. r'E:\msc_data\in100'   -- None = choose for me
MSC_ROOT = None      # e.g. r'E:\msc_results'        -- None = choose for me

# ---------------------------------------------------------------------------
import os

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_IN100_DIR'] = DATA_DIR
os.environ['MSC_SCRATCH'] = MSC_ROOT

PHASE = 'p1'
sess = M.Session(account='local', phase=PHASE, dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 worker_id=0, num_workers=1)

print()
print('layout under MSC_ROOT:')
print('  runs/{run_id}/  config.yaml  summary.json  STATUS.json')
print('                   metrics/     epochs.csv  final.csv  confusion_matrix.csv')
print('                                per_class.csv  exit_metrics.csv')
print('                   telemetry/   energy_samples.csv  system_samples.csv')
print('                                step_traces.jsonl')
print('                   per_sample/  test.parquet  train_holdout.parquet')
print('                                train_dynamics.parquet  meta.json')
print('                   checkpoints/ ckpt_last.pt  ckpt_best.pt')
print('                   env/         environment.json')
print('                   exit_heads.pt')
print('  budgets/{arch}.json     FLOPs per compute configuration')
print('  registry/events/*.jsonl  what ran, when, and how it ended')
print('  analysis/                Q1-Q4 outputs')
print('  tables/  paper/figures/  console/')

In [ ]:
# 03_INVENTORY.md claimed eight scores. It was an unverified claim: `msc` is
# not a stored column, and `el2n`/`forget_events` are NaN on the test split.
# Ask the artifact instead of trusting the document.
WANTED = ['msp', 'margin', 'entropy', 'ce_loss', 'el2n', 'forget_events',
          'pred_depth', 'msc']

import numpy as np, pandas as pd, itertools
from pathlib import Path

runs_dir = Path(MSC_ROOT) / 'runs'
runs = sorted(d.name for d in runs_dir.iterdir()
              if d.is_dir() and (d / 'per_sample' / 'test.parquet').exists())
meta = pd.DataFrame([{**M.parse_run_id(r), 'run_id': r} for r in runs])

_probe = pd.read_parquet(runs_dir / runs[0] / 'per_sample' / 'test.parquet')
absent = [c for c in WANTED if c not in _probe.columns]
allnan = [c for c in WANTED if c in _probe.columns
          and _probe[c].notna().mean() <= 0.5]
SCORES = [c for c in WANTED if c not in absent and c not in allnan]
if absent:
    print(f'not a column at all      : {absent}')
if allnan:
    print(f'NaN on the test split    : {allnan}  (training-set quantities)')
print(f'usable scores            : {SCORES}  ({len(SCORES)} of {len(WANTED)})')
if not SCORES:
    raise RuntimeError('no usable score columns -- refusing to continue')
base = meta[meta['method'] == 'base']
# ---- one run per (arch, seed) -------------------------------------------
# `resnet32x4` and `wrn_40_2` each have FIVE base runs: p0 pilots at seeds 1-2
# plus p1 at seeds 1-3, with byte-identical configs. So p0-s1 and p1-s1 are the
# SAME SEED run twice -- a replicate, which measures run-to-run nondeterminism,
# not seed variation. Pooling them conflated two different quantities and let
# two architectures supply 40 of 118 ordered pairs. Keep the highest phase.
_before = len(base)
base = (base.sort_values('phase')
            .drop_duplicates(subset=['arch', 'dataset', 'seed'], keep='last'))
if len(base) < _before:
    print(f'  dropped {_before - len(base)} duplicate (arch, seed) run(s) '
          f'-- pilot replicates; {len(base)} remain')
print(f'{len(base)} base run(s), {base["arch"].nunique()} architecture(s)')

---
## Routing, from the parquet alone

`pred_dk == label` gives per-exit correctness; `budgets/{arch}.json` gives the
cost ρ of each exit. Routing by any score is a sort on that column. No model is
needed, which is why this is minutes rather than GPU-hours.

**Lower score = route earlier**, so scores where *high* means *easy*
(`msp`, `margin`) are negated. The direction is asserted, not assumed — a
sign error here would invert the whole result.

In [ ]:
HIGH_MEANS_EASY = {'msp', 'margin'}      # everything else: high = hard

def exit_tables(run_id):
    d = pd.read_parquet(runs_dir / run_id / 'per_sample' / 'test.parquet')
    d = d.sort_values('sample_idx').reset_index(drop=True)
    ks = sorted(int(c.split('_d')[1]) for c in d.columns
                if c.startswith('pred_d') and c.split('_d')[1].isdigit())
    correct = np.stack([(d[f'pred_d{k}'].to_numpy() == d['label'].to_numpy())
                        for k in ks], axis=1).astype(float)
    conf = np.stack([d[f'top1p_d{k}'].to_numpy() for k in ks], axis=1)
    return d, correct, conf, ks

def _cost(k_assign, rho):
    return float(np.mean(np.asarray(rho)[k_assign]))

def _cost_of_counts(counts, rho):
    counts = np.asarray(counts, dtype=float)
    return float((counts * np.asarray(rho)).sum() / counts.sum())

def route_by(rank, correct, rho, target_rho, all_exits=True):
    '''Route each sample to an exit so the MEAN cost equals target_rho.
    Samples with the lowest `rank` exit earliest.

    all_exits=True  -- every one of the K exits is reachable (what an early-exit
                       system actually does).
    all_exits=False -- the original two-exit split (exit 0 or exit K-1) kept so
                       the first run's numbers remain reproducible.
    '''
    rank = np.asarray(rank, dtype=float)
    n, K = correct.shape
    u = np.empty(n)
    u[np.argsort(rank, kind='stable')] = np.arange(n) / max(n - 1, 1)
    lo, hi = 0.0, 1.0
    for _ in range(60):
        t = (lo + hi) / 2
        if all_exits:
            k_assign = np.clip((u * K * t * 2).astype(int), 0, K - 1)
        else:
            k_assign = np.where(u >= 1.0 - t, K - 1, 0)
        c = _cost(k_assign, rho)
        if c < target_rho: lo = t
        else: hi = t
    return float(correct[np.arange(n), k_assign].mean()), c

def route_confidence(conf, correct, rho, target_rho):
    '''The baseline the field actually deploys: exit at the FIRST exit whose
    top-1 probability clears a threshold. Crucially this reads confidence at
    the EARLY exit, so it is computable without running the rest of the net.

    The first version of this notebook used `-conf[:, -1]` -- the FINAL exit's
    confidence -- as the baseline. That needs a full forward pass to evaluate,
    so it is an oracle, not a baseline, and it made every headroom number a
    comparison between two oracles.
    '''
    n, K = correct.shape
    lo, hi = 0.0, 1.0
    for _ in range(60):
        th = (lo + hi) / 2
        fires = conf >= th
        fires[:, -1] = True                    # the last exit always answers
        k_assign = fires.argmax(axis=1)
        c = _cost(k_assign, rho)
        if c < target_rho: lo = th
        else: hi = th
    counts = np.bincount(k_assign, minlength=K)
    return float(correct[np.arange(n), k_assign].mean()), c, counts

def route_oracle(correct_choose, correct_eval, rho, target_rho):
    '''The real oracle ceiling at a budget, by Lagrangian relaxation.

    Choose an exit per sample to maximise expected correctness subject to a
    mean-cost constraint:   max_k  correct[i,k] - lambda * rho[k],  bisect
    lambda until the mean cost hits the budget. Because it is a maximum over
    EVERY assignment meeting the budget, it dominates any particular router --
    including a confidence threshold. That is what makes it a ceiling.

    Two earlier attempts were not ceilings and both produced negative headroom:
      1. sorting samples by a per-SAMPLE difficulty score, while the baseline
         thresholded per-EXIT confidence -- the baseline was better informed;
      2. forcing the oracle through the baseline's exit histogram -- filling
         exits greedily by cheapest-correct-exit is a heuristic, not the
         optimum, and canary 10 showed it losing by up to 3.2 points.

    `correct_choose` picks the exits, `correct_eval` scores them. Passing the
    same array gives the optimistic in-seed oracle; passing another seed's
    correctness gives the honest cross-seed one.
    '''
    rho = np.asarray(rho, dtype=float)
    n_ = correct_choose.shape[0]

    def assign(lam):
        return (correct_choose - lam * rho[None, :]).argmax(axis=1)

    lo, hi = 0.0, 100.0
    for _ in range(80):
        lam = (lo + hi) / 2
        if float(rho[assign(lam)].mean()) > target_rho: lo = lam
        else: hi = lam
    k = assign(hi)                       # cost <= target
    k_rich = assign(lo)                  # cost >= target

    # The argmax jumps in steps, so bisection typically lands UNDER budget --
    # canary 12 caught it at 0.608 against a target of 0.65. Leftover budget
    # understates the ceiling, so spend it: upgrade the samples with the best
    # correctness gain per unit of extra compute until the budget is used.
    idx = np.arange(n_)
    cand = np.nonzero(k_rich != k)[0]
    if len(cand):
        dcost = rho[k_rich[cand]] - rho[k[cand]]
        dgain = (correct_choose[cand, k_rich[cand]]
                 - correct_choose[cand, k[cand]])
        keep = dcost > 1e-12
        cand, dcost, dgain = cand[keep], dcost[keep], dgain[keep]
        if len(cand):
            room = (target_rho - float(rho[k].mean())) * n_
            for t in np.argsort(-(dgain / dcost), kind='stable'):
                if dcost[t] > room:
                    continue
                k[cand[t]] = k_rich[cand[t]]
                room -= dcost[t]
                if room <= 1e-12:
                    break
    return float(correct_eval[idx, k].mean()), float(rho[k].mean())

def oracle_rank(correct):
    '''The cheapest exit at which the sample is ACTUALLY CORRECT (K if never).

    This is what the early-exit literature means by an oracle -- "exit at the
    first layer whose prediction matches the final one" (08_RELATED_WORK.md S1)
    -- and it is per-EXIT information, K numbers per sample.

    Everything measured before this used a per-SAMPLE difficulty score as the
    "oracle" while the baseline thresholded per-EXIT confidence. The baseline
    knew "am I right at exit k"; the score only knew "is this sample generically
    hard". The baseline was strictly better informed, so the headroom came out
    at -8 accuracy points -- which is a real statement about difficulty scores,
    but is NOT a ceiling and must never be reported as one.
    '''
    correct = np.asarray(correct)
    n_, K_ = correct.shape
    ever = correct.any(axis=1)
    return np.where(ever, correct.argmax(axis=1), K_).astype(float)

def route_matched(rank, correct, counts):
    '''Route by `rank` using EXACTLY the exit histogram `counts`.

    Why this exists. A per-sample difficulty score carries ONE number per
    sample; confidence carries K (one per exit). So a confidence threshold can
    choose any exit histogram that meets the budget, while a sort on a
    difficulty score was being forced through a rigid quantile spread -- 20% of
    samples at every exit at rho=0.6. Comparing them then measures the
    MECHANISM, not the signal, and produced -10 accuracy points for every score
    at every budget: an oracle apparently losing to a threshold, which cannot
    happen.

    Fixing that means holding the mechanism constant. We take the baseline's
    own exit histogram and give the score the same one, so the cost is
    identical BY CONSTRUCTION -- no bisection, no residual budget mismatch --
    and the only thing that differs is WHICH samples go where. That is the
    question the study is actually asking.
    '''
    rank = np.asarray(rank, dtype=float)
    n = len(rank)
    order = np.argsort(rank, kind='stable')     # easiest first
    k_assign = np.empty(n, dtype=int)
    pos = 0
    for k, cnt in enumerate(counts):            # cheapest exit to the easiest
        cnt = int(cnt)
        k_assign[order[pos:pos + cnt]] = k
        pos += cnt
    if pos < n:
        k_assign[order[pos:]] = len(counts) - 1
    return float(correct[np.arange(n), k_assign].mean())

print('routing helpers defined -- correctness and cost come from the parquet')
print('  baseline = threshold on EARLY-exit confidence (deployable)')
print('  score routing = baseline exit histogram, samples chosen by score')
print('  TRUE oracle   = Lagrangian max over every assignment meeting the budget')

---
## The bias, both directions

R-03 in the risk register: seeds differ in accuracy, so routing seed *i* with
seed *j*'s score could look worse simply because *j* is a worse model. A real
optimism bias is **symmetric**; an accuracy confound is not. Both directions are
computed and reported.

In [ ]:
# The corpus is MIXED: 15 CIFAR-100 architectures and 2 ImageNet-100 ones.
# `sess.budgets(arch)` uses the SESSION's dataset, which paths_cell set to
# imagenet100 -- so it asked the imagenet zoo for `convnext_femto` and raised.
# A budget belongs to the RUN, not to the session.
_bud = {}
def rho_for(arch, dataset):
    if (arch, dataset) not in _bud:
        b = M.load_or_build_budgets(arch, sess.work, dataset)
        _bud[(arch, dataset)] = list(b['axes']['depth']['rho'])
    return _bud[(arch, dataset)]

TARGET_RHO = 0.80          # the operating point; the full curve comes next

rows, orows = [], []
for (arch, dset), grp in base.groupby(['arch', 'dataset']):
    ids = sorted(grp['run_id'])
    if len(ids) < 2:
        continue
    try:
        rho = rho_for(arch, dset)
    except Exception as e:
        print(f'  SKIP {arch} ({dset}): {type(e).__name__}: {str(e)[:70]}')
        continue
    tab = {r: exit_tables(r) for r in ids}
    for i, j in itertools.permutations(ids, 2):
        di, ci, confi, _ = tab[i]
        dj, _, _, _ = tab[j]
        common = di['sample_idx'].isin(dj['sample_idx']).to_numpy()
        base_conf, _, counts = route_confidence(confi[common], ci[common],
                                                rho, TARGET_RHO)

        # seed j's per-exit correctness, aligned onto seed i's common samples
        idx_common = di['sample_idx'].to_numpy()[common]
        posj = pd.Series(np.arange(len(dj)), index=dj['sample_idx'].to_numpy())
        cj_al = tab[j][1][posj.loc[idx_common].to_numpy()]

        # THE oracle: cheapest correct exit. In-seed = optimistic (it is scored
        # from the very model it routes); cross-seed = honest.
        a_in_true, c_in = route_oracle(ci[common], ci[common], rho, TARGET_RHO)
        a_cx_true, c_cx = route_oracle(cj_al,      ci[common], rho, TARGET_RHO)
        # An in-seed oracle knows this model's own correctness at every exit and
        # spends an identical budget. It cannot lose to a threshold on that same
        # model's confidence. If it does, the harness is broken, not the field.
        if a_in_true < base_conf - 1e-6 and c_in <= _cost_of_counts(counts, rho) + 1e-6:
            raise RuntimeError(
                f'{arch} {i}: in-seed oracle {a_in_true:.4f} < confidence '
                f'baseline {base_conf:.4f} at no greater cost '
                f'({c_in:.4f} vs {_cost_of_counts(counts, rho):.4f}). The '
                'oracle is a maximum over all assignments, so this is '
                'impossible -- the routing harness is wrong.')
        ci_c = ci[common]
        final_ok = ci_c[:, -1]
        early_ok = ci_c[:, :-1].max(axis=1)
        orows.append({'acc_full': float(final_ok.mean()),
                      'acc_best_exit': float(ci_c.mean(axis=0).max()),
                      'frac_early_saves': float(
                          ((early_ok > 0) & (final_ok == 0)).mean()),
                      'arch': arch, 'dataset': dset, 'model_seed': i[-2:],
                      'score_seed': j[-2:], 'oracle_in': a_in_true,
                      'oracle_cross': a_cx_true, 'baseline': base_conf,
                      'bias_true': a_in_true - a_cx_true,
                      'ceiling_honest': a_cx_true - base_conf,
                      'ceiling_optimistic': a_in_true - base_conf})
        for s in SCORES:
            sign = -1.0 if s in HIGH_MEANS_EASY else 1.0
            in_seed = sign * di[s].to_numpy(dtype=float)[common]
            cross   = sign * dj.set_index('sample_idx').loc[
                di['sample_idx'][common], s].to_numpy(dtype=float)
            if np.isnan(in_seed).all() or np.isnan(cross).all():
                continue
            a_in = route_matched(np.nan_to_num(in_seed, nan=np.inf),
                                 ci[common], counts)
            a_cx = route_matched(np.nan_to_num(cross, nan=np.inf),
                                 ci[common], counts)
            rows.append({'arch': arch, 'dataset': dset, 'score': s,
                         'model_seed': i[-2:],
                         'score_seed': j[-2:], 'in_seed': a_in,
                         'cross_seed': a_cx, 'bias': a_in - a_cx,
                         'msp_baseline': base_conf,
                         'headroom_honest': a_cx - base_conf})

bias = pd.DataFrame(rows)
M.save_analysis(sess.data_dir, 's2_optimism_bias', bias)

orc = pd.DataFrame(orows)
M.save_analysis(sess.data_dir, 's2_true_oracle', orc)
oc = orc[orc['dataset'] == 'cifar100']
print()
print('=== THE ORACLE CEILING (cheapest correct exit, matched budget) ===')
print(f'{len(oc)} (arch, seed-pair) rows, CIFAR-100, rho = {TARGET_RHO}')
# Levels and deltas printed separately. A median of differences is NOT the
# difference of medians -- the first version printed 78.30 %, 62.39 % and
# "+12.20 pt" together, and those do not subtract (78.30-62.39 = 15.91). A
# reader checking the arithmetic concludes the table is broken.
print('  medians of the LEVELS:')
print(f'    confidence baseline        : {oc["baseline"].median()*100:6.2f} %')
print(f'    oracle, in-seed            : {oc["oracle_in"].median()*100:6.2f} %')
print(f'    oracle, cross-seed (honest): {oc["oracle_cross"].median()*100:6.2f} %')
print('  medians of the PER-RUN DIFFERENCES (what the hypotheses test):')
print(f'    in-seed  - baseline        : '
      f'{oc["ceiling_optimistic"].median()*100:+6.2f} pt')
print(f'    cross-seed - baseline      : '
      f'{oc["ceiling_honest"].median()*100:+6.2f} pt')
print()
print(f'  OPTIMISM BIAS (in - cross) : '
      f'{oc["bias_true"].median()*100:+.3f} accuracy points')
print(f'  share of the apparent headroom that is optimism: '
      f'{100*oc["bias_true"].median()/max(oc["ceiling_optimistic"].median(),1e-9):.1f} %')

# ---- the reference line that decides whether any of this is real ---------
# An oracle that beats the network's OWN full-compute accuracy is not finding
# headroom; it is exploiting samples where an early exit happens to be right
# while the final layer is wrong. That is per-exit noise -- unavailable to any
# router, and non-transferable across seeds by definition. If the in-seed
# oracle sits above full compute, the optimism bias measures noise-harvesting,
# which IS the paper's claim, but it has to be shown rather than assumed.
print()
print('--- reference lines (same runs, same samples) ---')
print(f'  full compute, final exit   : {oc["acc_full"].median()*100:.2f} %')
print(f'  best single exit, no routing: {oc["acc_best_exit"].median()*100:.2f} %')
above = float((oc['oracle_in'] > oc['acc_full']).mean())
print(f'  in-seed oracle ABOVE full compute in {above*100:.0f}% of runs '
      f'(median {(oc["oracle_in"] - oc["acc_full"]).median()*100:+.2f} pt)')
print(f'  samples where an early exit is right and the FINAL is wrong: '
      f'{oc["frac_early_saves"].median()*100:.2f} %')
print('  ^ this is the pool the in-seed oracle harvests from.')

# If oracle_in is exactly acc_full + frac_early_saves, it is simply
# P(correct at ANY exit): the rho constraint is inactive, the oracle spends
# LESS than the baseline while scoring higher, and "matched FLOPs" is the wrong
# phrase. It stays a valid upper bound, and a conservative one -- the baseline
# at the oracle's lower cost would be worse still.
slack = float((oc['oracle_in'] - oc['acc_full']
               - oc['frac_early_saves']).abs().max())
if slack < 1e-9:
    print()
    print(f'  NOTE: the rho={TARGET_RHO} budget NEVER BINDS for the in-seed oracle.')
    print('  oracle_in == P(correct at ANY exit) exactly, on every run, so this')
    print('  is the UNCONSTRAINED bound and is not a matched-FLOPs comparison.')

print()
print('--- per architecture (is the bias driven by a few?) ---')
pa = (oc.groupby('arch')[['ceiling_optimistic', 'ceiling_honest', 'bias_true']]
        .median() * 100)
print(pa.round(2).sort_values('bias_true', ascending=False).to_string())
print(f'{len(bias)} (arch, score, seed-pair) rows')
print(bias.groupby('score')[['bias', 'headroom_honest']].mean().round(4).to_string())

---
## R3 — is the in-seed oracle optimistic?

**H3:** median bias ≥ **0.5 accuracy points** and > 0 for at least 6 of 8 scores.

Either answer is reportable: a large bias means the field's oracle bounds are
inflated; a bias of zero validates a practice nobody had checked.

In [ ]:
med = bias['bias'].median() * 100
per_score = bias.groupby('score')['bias'].median() * 100
n_pos = int((per_score > 0).sum())

print(per_score.round(3).sort_values(ascending=False).to_string())
print()
print(f'median bias over the whole grid : {med:+.3f} accuracy points')
print(f'scores with positive bias       : {n_pos} of {len(per_score)}')
print(f'H3 (>= 0.5 pt AND >= 6 of 8)    : '
      f'{"SUPPORTED" if (med >= 0.5 and n_pos >= 6) else "NOT SUPPORTED"}')
print()
print('symmetry check (R-03): a real bias is direction-symmetric;')
print('an accuracy confound is not.')
sym = bias.groupby(['arch', 'score']).apply(
    lambda g: g['bias'].std(), include_groups=False)
print(f'  mean within-pair sd of bias: {sym.mean()*100:.3f} pt')

---
## R5 — after correction, is there headroom?

**H5:** no score's **cross-seed** oracle beats `msp` by more than 1.0 point.

This is the gate. If none clears it, no method is built and the bound is the
result (`02_PROTOCOL.md` stopping rule 2).

In [ ]:
hon = bias.groupby('score')['headroom_honest'].median() * 100
print(hon.round(3).sort_values(ascending=False).to_string())
n = int(bias['arch'].nunique())
se2 = 2 * (0.5 / np.sqrt(10000)) * 100      # 2 SE on an accuracy diff, 10k samples
print()
print(f'noise floor (2 SE, 10k samples): +/-{se2:.3f} pt')
print(f'best honest headroom           : {hon.max():+.3f} pt  ({hon.idxmax()})')
print(f'H5 (nothing clears +1.0 pt)    : '
      f'{"SUPPORTED -- no method is built" if hon.max() < 1.0 else "FALSIFIED"}')
# ---- what clearing the gate does and does NOT license -------------------
# `pred_depth` is not a deployable routing signal. prediction_depth() runs a
# kNN probe over the features of EVERY layer and targets the network's own
# final answer, so obtaining it costs a full forward pass -- a router that
# needs the whole network to decide where to stop saves nothing. It is the
# textbook Oracle-EE rule (08_RELATED_WORK.md S1) wearing a score's clothes.
#
# So a large number here is a CEILING, not a method. Read it as: this much
# accuracy is on the table at this budget for a router that could predict
# prediction depth from cheap early features. That is the follow-up question,
# and it is worth asking precisely because the ceiling is not flat.
ORACLE_ONLY = {'pred_depth'}
deployable = hon.drop(index=[s_ for s_ in ORACLE_ONLY if s_ in hon.index])

print()
print(f'  oracle-only signals (need full compute to evaluate): {sorted(ORACLE_ONLY)}')
print(f'  best DEPLOYABLE headroom : {deployable.max():+.3f} pt  ({deployable.idxmax()})')
print(f'  ceiling from the oracle  : {hon.max():+.3f} pt  ({hon.idxmax()})')
if hon.max() >= 1.0 and deployable.max() < 1.0:
    print()
    print('  -> The GATE IS AMBIGUOUS and must not be read as "build a method".')
    print('     No deployable score clears +1.0; the oracle clears it by a lot.')
    print('     That is a statement about headroom, not about any method, and')
    print('     it contradicts Study 1 B11 (+0.00007) -- which used MSC, a')
    print('     cost-normalised aggregate, where this uses the raw per-sample')
    print('     sufficient depth. Chase the discrepancy before building.')

---
## R4 — does the bias follow reliability?

**H4:** bias correlates with (1 − ρ_seed) at Spearman ≥ 0.5.

If it holds, two observations become one mechanism — and ρ_seed becomes a cheap
predictor of how inflated a published oracle bound is.

n is small. The scatter is reported, not just the coefficient.

---
## R-04 — the operating point, not one point

A ceiling measured at a single budget is one point on a curve. Study 1's B11
lived at rho = 0.806 and concluded there was no headroom anywhere.

In [ ]:
sweep = []
for tr in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    for (arch, dset), grp in base.groupby(['arch', 'dataset']):
        if dset != 'cifar100':
            continue
        ids = sorted(grp['run_id'])
        if len(ids) < 2:
            continue
        try:
            rho = rho_for(arch, dset)
        except Exception:
            continue
        i, j = ids[0], ids[1]
        di, ci, confi, _ = exit_tables(i)
        dj, _, _, _ = exit_tables(j)
        common = di['sample_idx'].isin(dj['sample_idx']).to_numpy()
        b, _, cnts = route_confidence(confi[common], ci[common], rho, tr)
        for sc in SCORES:
            sign = -1.0 if sc in HIGH_MEANS_EASY else 1.0
            cross = sign * dj.set_index('sample_idx').loc[
                di['sample_idx'][common], sc].to_numpy(dtype=float)
            if np.isnan(cross).all():
                continue
            a = route_matched(np.nan_to_num(cross, nan=np.inf), ci[common], cnts)
            sweep.append({'target_rho': tr, 'arch': arch, 'score': sc,
                          'headroom': (a - b) * 100})

sw = pd.DataFrame(sweep)
M.save_analysis(sess.data_dir, 's2_headroom_sweep', sw)
piv = sw.groupby(['target_rho', 'score'])['headroom'].median().unstack()
print('median honest headroom (accuracy points) vs compute budget')
print(piv.round(2).to_string())
print()
print('If the curve is flat everywhere, the bound is the paper. If headroom')
print('appears in a region, that region becomes the subject (R-04).')

In [ ]:
from scipy.stats import spearmanr
grid = pd.read_csv(Path(sess.data_dir) / 'analysis' / 's2_reliability_grid.csv')
grid = grid[(grid['dataset'] == 'cifar100') & (grid['split'] == 'test')]  # D3
j = (bias[bias['dataset'] == 'cifar100']
     .groupby(['arch', 'score'])['bias'].median().reset_index()
     .merge(grid[['arch', 'score', 'rho_seed']], on=['arch', 'score']))
j['unreliability'] = 1 - j['rho_seed']
M.save_analysis(sess.data_dir, 's2_bias_vs_reliability', j)

m = j[['unreliability', 'bias']].dropna()
r, p = spearmanr(m['unreliability'], m['bias'])
print(f'n = {len(m)} (arch, score) cells')
print(f'Spearman(1 - rho_seed, bias) = {r:+.3f}   p = {p:.4f}')
print(f'H4 (>= 0.5): {"SUPPORTED" if r >= 0.5 else "NOT SUPPORTED"}')

# The test above uses the per-SCORE bias. The ORACLE bias is the study's actual
# quantity, so test that too rather than letting the reader assume they agree.
try:
    orc2 = pd.read_csv(Path(sess.data_dir) / 'analysis' / 's2_true_oracle.csv')
    orc2 = orc2[orc2['dataset'] == 'cifar100']
    rel = (grid[grid['score'] == 'ce_loss'][['arch', 'rho_seed']]
           .drop_duplicates('arch'))
    jj = (orc2.groupby('arch')['bias_true'].median().reset_index()
          .merge(rel, on='arch'))
    r2, p2 = spearmanr(1 - jj['rho_seed'], jj['bias_true'])
    print()
    print(f'  same test on the ORACLE bias: rho = {r2:+.3f}  p = {p2:.4f}  '
          f'(n = {len(jj)} architectures)')
except Exception as e:
    print(f'  [oracle-bias variant skipped: {type(e).__name__}: {e}]')
print()
print('per-score medians (the scatter behind the number):')
print(j.groupby('score')[['rho_seed', 'bias']].median().round(4).to_string())